In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

/opt/anaconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

True

In [7]:
llm=ChatGroq(model="llama-3.1-8b-instant")
parser=StrOutputParser()
prompt=ChatPromptTemplate([
    ("system","you are a helpful assistant"),
    MessagesPlaceholder(variable_name="history"),
    ("human","{input}")])
chain=prompt|llm|parser
store={}
def get_session_history(session_id:str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]


In [8]:
chain_with_memory=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history")

/opt/anaconda3/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3701: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [16]:
def chat(message,session_id="session_1"):
    response=chain_with_memory.invoke(
        {"input":message},
        config={"configurable": {"session_id": session_id}})
    print(f"You: {message}")
    print(f"AI : {response}")
    print()
    return response
print("═" * 50)
print("FILE 1: Buffer Memory — stores everything")
print("═" * 50)
print()

══════════════════════════════════════════════════
FILE 1: Buffer Memory — stores everything
══════════════════════════════════════════════════



In [23]:
# ── Updated Chat Questions for LinkedIn Showcase ──

# Turn 1: Setting up a complex development context
chat("Hi! I'm Krish. I'm building an production-ready RAG application")

# Turn 2: Providing specific technical hurdles or goals
chat("Right now, I'm aiming for an 82% recall metric on a smart triage system, but latency is a bit high.")

# Turn 3: Testing if buffer memory recalls the user's specific hardware/setup
chat("What machine am I using to build my app?")

# Turn 4: Testing deep retrieval of the specific metric mentioned earlier
chat("What was the target recall percentage I mentioned?")

# Turn 5: Testing if it connects the dots of your overall engineering goal
chat("Summarize my project objective and what I'm trying to optimize.")

You: Hi! I'm Krish. I'm building an production-ready RAG application
AI : Hello Krish, nice to chat with you again. You're working on building a production-ready RAG (Retrieval-Augmented Generation) application. What's your current focus or challenge with the project?

You: Right now, I'm aiming for an 82% recall metric on a smart triage system, but latency is a bit high.
AI : You're aiming to achieve an 82% recall metric on a smart triage system using your RAG application, but you're experiencing high latency. That's a good goal, but latency is a critical issue in a production-ready system. How can I help you optimize the latency while still achieving your recall metric?

You: What machine am I using to build my app?
AI : You're building your RAG application on a Mac M3.

You: What was the target recall percentage I mentioned?
AI : You mentioned that your target recall percentage is 82%.

You: Summarize my project objective and what I'm trying to optimize.
AI : Here's a summary:

**Pr

"Here's a summary:\n\n**Project Objective:** Building a production-ready RAG application for a smart triage system.\n\n**Goal:** Achieve a recall metric of 82% on the system.\n\n**Current Challenge:** Optimizing high latency in the system."

In [24]:
# ── Inspect raw memory ────────────────────────────────────
print("═" * 50)
print("What is stored in memory right now:")
print("═" * 50)
history = store["session_1"]
for msg in history.messages:
    print(f"{msg.type.upper()}: {msg.content}")
    print()

══════════════════════════════════════════════════
What is stored in memory right now:
══════════════════════════════════════════════════
HUMAN: Hi! My name is Krish and I am from Ahmedabad.

AI: Hello Krish, it's nice to meet you. Ahmedabad is a great city in Gujarat, India, known for its rich history and cultural heritage. What brings you here today, and is there anything I can help you with?

HUMAN: I am learning LangChain to get a ₹50k internship.

AI: LangChain is a powerful framework for building conversational AI models. It's great that you're learning it to pursue a ₹50k internship.

LangChain allows you to build conversational agents that can engage in natural-sounding conversations with users. It's a popular choice among developers and researchers working on conversational AI projects.

To help you achieve your goal, can you tell me a bit more about your current level of experience with LangChain? Have you completed any tutorials or projects yet?

HUMAN: What is my name?

AI:

In [25]:

print(f"Total messages stored: {len(history.messages)}")
print("Grows by 2 every turn (1 human + 1 AI)")
print("This is the problem with buffer memory in production")

Total messages stored: 40
Grows by 2 every turn (1 human + 1 AI)
This is the problem with buffer memory in production
